# Meeting minutes creator

In this colab, we make a meeting minutes program.

It includes useful code to connect your Google Drive to your colab.

Upload your own audio to make this work!!

https://colab.research.google.com/drive/1KSMxOCprsl1QRpt_Rq0UqCAyMtPqDQYx?usp=sharing

This should run nicely on a low-cost or free T4 box.

In [9]:
from mistralai.client import Mistral
import dotenv
import os
import requests 
from transformers import pipeline

dotenv.load_dotenv()
key = os.getenv("MISTRAL_API_KEY")
hf_token = os.getenv("HUGGING_FACE")
client = Mistral(api_key=key)

In [6]:
payload = {"model": "mistral-small-latest","messages": [{"role": "user", "content": "Hello"}],"temperature": 0,
                   "logprobs": True,  "stream": True}

response = requests.post("https://api.mistral.ai/v1/chat/completions",headers=
                                 {"Authorization": f"Bearer {key}"},json=payload)

In [7]:
# Logprobs do not work with mistral :(
response.content

b'{"object":"error","message":"Logprobs are not enabled for this model","type":"invalid_request_invalid_args","param":null,"code":"3051","raw_status_code":400}'

### Now trying the excersize: transcribing through Whisper.



In [13]:
pipe = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device='cuda')

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [14]:
file_loc = "C:/Users/teede/Downloads/denver_extract.mp3"
output = pipe(file_loc, return_timestamps=True)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
output

{'text': " kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so let me see the creation of the logo here. yeah so that basically um kind of sums up the reason behind the logo and the um all the meanings behind the symbolism and uh i'll you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegian

: 

In [24]:
model ="mistral-small-latest"

In [20]:
SYSTEM_PROMPT = """You will receive a recording of the city council. \n
Please make a summary of the meeting. The input you will receive consists out of a json \n
with text and a timestamp. You can ignore the timestamps, only focus on the text.""" 



In [18]:
USER_PROMPT  = f"This is the content:{output}"

In [19]:
USER_PROMPT

'This is the content:{\'text\': " kind of the confluence of this whole idea of the confluence week, the merging of two rivers, and as we\'ve kind of seen recently in politics and in the world, there\'s a lot of situations where water is very important right now, and it\'s a very big issue, so that is the reason that the back of the logo is considered water, so let me see the creation of the logo here. yeah so that basically um kind of sums up the reason behind the logo and the um all the meanings behind the symbolism and uh i\'ll you\'ll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Council

In [32]:
response = client.chat.complete(model=model, messages = [{"role":"system", "content":SYSTEM_PROMPT},
                                            {"role":"user","content": USER_PROMPT}])

In [38]:
from IPython.display import display, Markdown, clear_output

In [39]:
Markdown(response.choices[0].message.content)

### **Summary of the Denver City Council Meeting – October 9, 2017**

#### **1. Opening & Roll Call**
- The meeting began with the Pledge of Allegiance led by Councilman Lopez.
- Roll call confirmed all 11 council members were present, establishing a quorum.
- Minutes from the October 2nd meeting were approved without corrections.

#### **2. Council Announcements**
- **Councilman Clark** announced the first-ever **Halloween Parade on Broadway** (District 7) on **October 21st at 6 PM**, featuring costumes, candy, and themed floats.

#### **3. Proclamation on Indigenous Peoples Day**
- **Proclamation 1127** was presented, officially designating **October 9, 2017, as Indigenous Peoples Day in Denver**.
- The proclamation recognized:
  - Indigenous peoples' historical and ongoing contributions to Denver.
  - The ancestral lands of tribes like the Southern Ute, Ute Mountain Ute, Arapaho, and Cheyenne.
  - Denver’s designation of Indigenous Peoples Day in 2016 (replacing Columbus Day).
  - Efforts by Indigenous youth in events like **Confluence Week** and leadership conferences.
- **Councilman Lopez** delivered an emotional speech, emphasizing pride in Denver’s Indigenous heritage while acknowledging ongoing struggles (poverty, lack of services).
- **Councilwoman Ortega** and **Councilwoman Canich** praised the proclamation, highlighting cultural preservation and environmental activism by Indigenous communities.
- The proclamation was adopted unanimously.

#### **4. Key Themes Discussed**
- **Symbolism of the Logo**: The design reflects the confluence of rivers (Denver’s founding) and water as a vital resource.
- **Cultural Pride vs. Disrespect**: Speakers emphasized celebrating Indigenous culture without diminishing others.
- **Social Justice**: Acknowledgment of systemic issues affecting Indigenous communities today.
- **National Recognition**: Denver was highlighted in *Time* and *Newsweek* for its Indigenous Peoples Day observance.

#### **5. Closing Remarks**
- The meeting underscored Denver’s commitment to inclusivity and honoring Indigenous history while addressing contemporary challenges.

**Final Note**: The council framed Indigenous Peoples Day as a **"day on"**—a call to action for equity and recognition, not just a day off.